# **Download Dependencies (a100 @ 6/22/26)**

In [1]:
import sys

# Uninstall potentially conflicting packages to ensure a clean slate
!pip uninstall -y vllm triton torch torchvision torchaudio Pillow tensorflow tensorflow-io

# Install torch and torchvision specifically for CUDA 12.x (common in Colab)
!pip install torch==2.4.0 torchvision==0.19.0 torchaudio==2.4.0 --index-url https://download.pytorch.org/whl/cu121 --quiet

# Install vllm allowing it to pick a compatible version, with transformers pinned to 4.44.2
!pip install vllm==0.6.0 transformers==4.44.2 --quiet

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: pillow 11.3.0
Uninstalling pillow-11.3.0:
  Successfully uninstalled pillow-11.3.0
Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 116.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 120.9 MB/s eta 0:00:00


In [1]:
import os

def get_secret(key_name):
    """Fetch a secret from Colab, Kaggle, or env — whichever is available."""
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key_name)
    except Exception:
        pass
    return os.getenv(key_name)

hf_token = get_secret("HF_TOKEN")
assert hf_token, "HF_TOKEN not found — add it in Kaggle > Add-ons > Secrets"

# Log in so vLLM can pull the gated model
from huggingface_hub import login, HfApi
login(token=hf_token)

# Instantiate the Hugging Face API client
api = HfApi(token=hf_token)

# **Snapshot Data**

In [2]:
!mkdir -p ARL
base="https://github.com/James-Tiny-Tjib/ARL/raw/main/ARL"
!curl -L -o "ARL/master_snapshots_1000.parquet" "$base/master_snapshots_1000.parquet"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  501k  100  501k    0     0  2917k      0 --:--:-- --:--:-- --:--:-- 2917k


## **Prepare Snapshot String**



In [3]:
from datasets import load_dataset
import numpy as np
import torch

snapshot_dataset = load_dataset("parquet", data_files = "ARL/master_snapshots_1000.parquet")

def softmax(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - x.max())
    return e / e.sum()

def generate_snapshot_string(index):
    """
    Index conventions in the snapshot:
      fusion_confidence: [P(friendly), P(threat)]
      rf_logits:         [friendly,    threat]      -> softmax
      au_logits:         [mambo,       bebop, background]  -> softmax
      vs_logits:         scalar in [0,1] = P(drone)
    """
    snap = snapshot_dataset["train"][index]

    # # ── Fusion (system-wide) ────────────────────────────────────────────
    # fusion_threat = float(snap["fusion_confidence"][1])
    # if fusion_threat >= 0.5:
    #     output = (f"The model is {fusion_threat*100:.2f}% confident there is "
    #               f"a drone threat within the entire campus.\n")
    # else:
    #     output = (f"The model is {(1 - fusion_threat)*100:.2f}% confident there is "
    #               f"no drone threat within the entire campus.\n")

    output = ""

    # ── Per-sensor ──────────────────────────────────────────────────────
    for idx, s in enumerate(snap["sensor_list"]):
        lines = [f"Sensor {idx} data:"]

        # RF
        rf = softmax(s["rf_logits"])
        rf_threat = float(rf[1])
        if rf_threat >= 0.5:
            lines.append(f"\tRadio Frequency (RF) Detection: {rf_threat*100:.2f}% drone threat")
        else:
            lines.append(f"\tRadio Frequency (RF) Detection: {(1 - rf_threat)*100:.2f}% no drone threat")

        # Audio  (threat = mambo + bebop)
        au = softmax(s["au_logits"])
        au_threat = float(au[0] + au[1])
        if au_threat >= 0.5:
            lines.append(f"\tAudio Detection: {au_threat*100:.2f}% drone threat")
        else:
            lines.append(f"\tAudio Detection: {(1 - au_threat)*100:.2f}% no drone threat")
        lines.append("\tDrone Type:")
        lines.append(f"\t\tMambo (drone):         {au[0]*100:.2f}%")
        lines.append(f"\t\tBebop (drone):         {au[1]*100:.2f}%")
        lines.append(f"\t\tBackground (no drone): {au[2]*100:.2f}%")

        # Visual (already P(drone))
        vs = float(s["vs_logits"])
        if vs >= 0.5:
            lines.append(f"\tVisual Detection: {vs*100:.2f}% drone threat")
        else:
            lines.append(f"\tVisual Detection: {(1 - vs)*100:.2f}% no drone threat")

        output += "\n".join(lines) + "\n"
    return output

print(snapshot_dataset["train"][5])
print(generate_snapshot_string(5))


Generating train split: 0 examples [00:00, ? examples/s]

{'timestamp': 1782096051.1770785, 'drone_pos': [338, 401], 'is_threat_gt': 0, 'nearest_sensor': 15, 'nearest_sensor_dist': 95.12623192369179, 'fusion_confidence': [0.2409355640411377, 0.7590644359588623], 'sensor_list': [{'x': 235, 'y': 37, 'triggered': False, 'audio_triggered': False, 'rf_logits': [0.297666072845459, -0.2953551113605499], 'au_logits': [1.8378545045852661, -6.914989948272705, 6.7876129150390625], 'vs_logits': 4.540453301160596e-05}, {'x': 190, 'y': 89, 'triggered': False, 'audio_triggered': False, 'rf_logits': [0.8029138445854187, -0.7960835695266724], 'au_logits': [-0.9022834897041321, -4.965112686157227, 8.246222496032715], 'vs_logits': 0.0022620437666773796}, {'x': 167, 'y': 135, 'triggered': False, 'audio_triggered': False, 'rf_logits': [0.5480042695999146, -0.539214015007019], 'au_logits': [1.2199302911758423, -6.261110782623291, 7.08050012588501], 'vs_logits': 6.558851509907981e-06}, {'x': 190, 'y': 221, 'triggered': False, 'audio_triggered': False, 'rf_logits': 

# **Generate Responses**

In [4]:
import os
import re
import json
import math
import random
import unicodedata
from collections import defaultdict
from datasets import load_from_disk, Dataset
from datasets import load_dataset
import json
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
# ============================================================================
# MODELS AND DATASETS
# ============================================================================

llm = LLM(
    model="Qwen/Qwen2.5-32B-Instruct-AWQ",
    quantization="awq_marlin",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    tensor_parallel_size=1,
    trust_remote_code=True,
)

tokenizer = llm.get_tokenizer()

query_dataset = load_dataset("JamesResearch1216/threat-detection-queries-v3", split = "train")

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 06-23 17:58:01 awq_marlin.py:89] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-23 17:58:01 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_mod

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


INFO 06-23 17:59:02 model_runner.py:926] Loading model weights took 18.1477 GB
INFO 06-23 17:59:06 gpu_executor.py:122] # GPU blocks: 12964, # CPU blocks: 1024
INFO 06-23 17:59:08 model_runner.py:1217] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-23 17:59:08 model_runner.py:1221] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-23 17:59:39 model_runner.py:1335] Graph capturing finished in 31 secs.


README.md:   0%|          | 0.00/345 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/321k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9999 [00:00<?, ? examples/s]

In [ ]:
"""
generate_responses.py
=====================
STAGE 4 of the pipeline. Given the v3 query dataset and the snapshot dataset,
ask the teacher model (Qwen-32B-AWQ) to produce a faithful response for each
(query, snapshot) pair, and save the result as a labeled HF dataset.

Inputs (assumed to be in scope from earlier notebook cells):
    llm                       # vLLM engine
    tokenizer                 # from llm.get_tokenizer()
    snapshot_dataset["train"] # parquet of ~1000 master snapshots
    query_dataset             # JamesResearch1216/threat-detection-queries-v3 ("train" split)
    generate_snapshot_string  # the index->string formatter (already fixed)
    hf_token                  # Hugging Face token

Output schema (the 4 things you asked for, plus metadata for traceability):
    query           : the user's question
    snapshot_string : formatted sensor readings
    instruction     : the CONSTANT system prompt (same string in every row)
    response        : the teacher's answer
    persona, intent, snapshot_idx : metadata
"""

# ============================================================================
# KNOBS
# ============================================================================
MAX_QUERIES        = 10000        # int to cap (e.g. 100 for a smoke test); None = all
UPDATE_HF_EVERY    = 160
BATCH_SIZE         = 16
SEED               = 67
HF_USERNAME        = "JamesResearch1216"
HF_QUERY_DATASET   = "JamesResearch1216/threat-detection-queries-v4"
HF_REPO_NAME       = "threat-detection-responses-10k"
PUSH_PRIVATE       = False
JSONL_CHECKPOINT   = "./responses_partial.jsonl"   # written incrementally so a crash doesn't lose hours

# Sampling
SAMPLING_KW = dict(
    temperature=0.7,         # faithful + slightly varied; not as high as query-gen
    top_p=0.95,
    max_tokens=400,          # responses should be short-to-medium; cap to keep batches fast
    frequency_penalty=0.1,
)



# ============================================================================
# THE CONSTANT INSTRUCTION
# Same string is used by the child model at inference time. It needs to be
# self-contained: sensor layout, modalities, spatial groupings, response style.
# ============================================================================
INSTRUCTION = """\
You are the AI assistant for an airport counter-drone threat-detection system. \
Operators, responders, and members of the public ask you what is happening — \
sometimes formally, sometimes anxiously, sometimes casually — and you give them \
a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, \
indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), \
or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide \
threat estimate. You receive that estimate at the top of each snapshot, then \
the per-sensor breakdown.

SENSOR GROUPINGS BY REGION
When the user asks about an area, use these named groups:
- Quadrants:
  - First quadrant:  sensors 11-16
  - Second quadrant: sensors 5-11
  - Third quadrant:  sensors 0-5
  - Fourth quadrant: sensors 16-22
- Hemispheres:
  - North: sensors 5-16
  - South: sensors 0-5 and 16-22
  - East:  sensors 11-22
  - West:  sensors 0-11

HOW TO ANSWER
- Ground every claim in the snapshot. NEVER invent numbers, sensors, or \
readings the snapshot does not show.
- Match the user's voice. A commander gets a clipped, direct answer; a worried \
bystander gets reassurance in plain words; a casual user gets a casual reply.
- For lay users (worried bystanders, casual askers), do NOT use technical words \
like "logit", "softmax", "modality", or "confidence vector". Use natural \
phrases: "the camera is picking it up", "the radio is quiet", "high confidence", \
"I'm not sure yet".
- Do not dump the entire snapshot. Focus on what the user asked.
- When several sensors or modalities agree, say so — agreement is stronger \
evidence than a single hit.
- When asked about direction or area, use the named groupings above.
- Do not offer to display images, play audio, pull spectrograms, or take any \
action outside answering the question. You can only report on what the \
sensors are seeing.
- Be honest about uncertainty. If a reading is borderline (around 50%), say \
so rather than overcommitting.
- Keep responses appropriately brief. One to four sentences is usually right; \
a formal sitrep may justify a little more, a casual ping may justify a lot less."""


# ============================================================================
# USER-TURN MESSAGE  (query + snapshot)
# ============================================================================
def make_user_message(query, snapshot_string):
    return (f"USER QUESTION:\n{query}\n\n"
            f"CURRENT SENSOR READINGS:\n{snapshot_string}")


def build_chat_prompt(query, snapshot_string, tokenizer):
    messages = [
        {"role": "system", "content": INSTRUCTION},
        {"role": "user",   "content": make_user_message(query, snapshot_string)},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


# ============================================================================
# CHECKPOINT  (so a kernel crash mid-run doesn't cost hours of generation)
# ============================================================================
def load_done_query_indices(path):
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path, "r") as f:
        for line in f:
            try:
                row = json.loads(line)
                if "query_idx" in row:
                    done.add(row["query_idx"])
            except json.JSONDecodeError:
                continue
    return done


def append_jsonl(path, row):
    with open(path, "a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


# ============================================================================
# DRIVER  — uses notebook-scope `llm`, `tokenizer`, `snapshot_dataset`,
#           `query_dataset`, `generate_snapshot_string`, `hf_token`.
# ============================================================================
def run(llm, tokenizer, snapshot_dataset, query_dataset,
        generate_snapshot_string, hf_token,
        max_queries=MAX_QUERIES, resume=True):

    random.seed(SEED)

    # ── 1) Pre-format every snapshot once (1k strings; reused ~10x each) ───
    n_snaps = len(snapshot_dataset["train"])
    print(f"Pre-formatting {n_snaps} snapshot strings ...")
    snap_strings = [generate_snapshot_string(i) for i in range(n_snaps)]
    avg_len = sum(len(s) for s in snap_strings) / max(1, len(snap_strings))
    print(f"  ✓ {n_snaps} strings ready (avg {avg_len:,.0f} chars each)")

    # ── 2) Pair every query with a uniformly-random snapshot (seeded) ──────
    n_queries = len(query_dataset)
    if max_queries is not None:
        n_queries = min(n_queries, max_queries)
    pairings = [(qi, random.randint(0, n_snaps - 1)) for qi in range(n_queries)]
    print(f"Paired {n_queries} queries to snapshots "
          f"(each snapshot reused ~{n_queries / n_snaps:.1f}x on average).")

    # ── 3) Resume support ─────────────────────────────────────────────────
    done = load_done_query_indices(JSONL_CHECKPOINT) if resume else set()
    if done:
        print(f"Resuming: {len(done):,} queries already in {JSONL_CHECKPOINT}, "
              f"skipping those.")
    todo = [(qi, si) for (qi, si) in pairings if qi not in done]
    print(f"To generate: {len(todo):,} queries.")

    # ── 4) Build prompts ──────────────────────────────────────────────────
    prompts = [
        build_chat_prompt(query_dataset[qi]["query"], snap_strings[si], tokenizer)
        for (qi, si) in todo
    ]

    # quick token-length sanity
    if prompts:
        sample_tok = tokenizer(prompts[0])["input_ids"]
        print(f"Sample prompt: {len(sample_tok):,} input tokens.")

    # ── 5) Generate, batched, with incremental save ───────────────────────
    sampling_params = SamplingParams(**SAMPLING_KW)
    n_batches = math.ceil(len(prompts) / BATCH_SIZE)
    num_batches = 0
    chunk_rows = []
    for bi, start in enumerate(range(0, len(prompts), BATCH_SIZE), start=1):
        end = min(start + BATCH_SIZE, len(prompts))
        batch_prompts = prompts[start:end]
        batch_meta = todo[start:end]

        outs = llm.generate(batch_prompts, sampling_params)
        for out, (qi, si) in zip(outs, batch_meta):
            q = query_dataset[qi]
            # The row data
            row_data = {
                "query_idx":    qi,
                "snapshot_idx": si,
                "query":        q["query"],
                "persona":      q.get("persona", ""),
                "intent":       q.get("intent", ""),
                "response":     out.outputs[0].text.strip(),
            }

            # Save incrementally for crash recovery
            append_jsonl(JSONL_CHECKPOINT, row_data)

            # Add HF formatting requirements and append to our memory list
            row_data["snapshot_string"] = snap_strings[si]
            row_data["instruction"] = INSTRUCTION
            chunk_rows.append(row_data)


        num_batches += 1
        # Upload every 10 batches
        if num_batches % 10 == 0:

            repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"

            api.create_repo(
                repo_id=repo_id,
                repo_type="dataset",
                exist_ok=True,           # Prevents errors if the dataset already exists
                private=False            # Set to True if you want a private dataset
            )
            print(f"\nSending Batches {num_batches-10} - {num_batches} to HF")

            # Convert list directly to HuggingFace Dataset
            ds = Dataset.from_list(chunk_rows)
            parquet_filename = f"train-{num_batches-10:05d}.parquet"
            ds.to_parquet(parquet_filename)

            # FIX 2: Use the initialized HfApi object
            api.upload_file(
                path_or_fileobj=parquet_filename,
                path_in_repo=parquet_filename,
                repo_id=repo_id,
                repo_type="dataset",
                token=hf_token,
            )
            print(f"  ✓ https://huggingface.co/datasets/{repo_id}")

            # FIX 3: Clear the list so the next chunk starts fresh
            chunk_rows = []

        if bi % 10 == 0 or bi == n_batches:
            done_so_far = len(done) + end
            print(f"  batch {bi}/{n_batches} done ({done_so_far:,}/{n_queries:,} queries)")

    # ── 6) Build the final dataset from JSONL ─────────────────────────────
    print(f"\nBuilding final dataset from {JSONL_CHECKPOINT} ...")
    rows = []
    with open(JSONL_CHECKPOINT) as f:
        for line in f:
            r = json.loads(line)
            rows.append({
                "query":           r["query"],
                "snapshot_string": snap_strings[r["snapshot_idx"]],
                "instruction":     INSTRUCTION,
                "response":        r["response"],
                "persona":         r.get("persona", ""),
                "intent":          r.get("intent", ""),
                "snapshot_idx":    r["snapshot_idx"],
            })
    ds = Dataset.from_list(rows)
    ds.save_to_disk("./responses_ds")
    ds.to_json("./responses.jsonl")
    print(f"  ✓ {len(ds):,} rows saved locally to ./responses_ds and ./responses.jsonl")

    # ── 7) Push to HF Hub ─────────────────────────────────────────────────
    # if hf_token:
    #     repo_id = f"{HF_USERNAME}/{HF_REPO_NAME}"
    #     print(f"Pushing to {repo_id} ...")
    #     ds.push_to_hub(repo_id, token=hf_token, private=PUSH_PRIVATE)
    #     print(f"  ✓ https://huggingface.co/datasets/{repo_id}")

    # ── 8) Quick report ───────────────────────────────────────────────────
    from collections import defaultdict
    by_intent = defaultdict(int); by_persona = defaultdict(int)
    for r in rows:
        by_intent[r["intent"]] += 1; by_persona[r["persona"]] += 1
    print("\nIntent coverage:")
    for k in sorted(by_intent): print(f"  {k:<28} {by_intent[k]:>5}")
    print("\nPersona coverage:")
    for k in sorted(by_persona): print(f"  {k:<20} {by_persona[k]:>5}")
    print("\nSample rows:")
    for r in random.sample(rows, k=min(3, len(rows))):
        print(f"\n[{r['persona']} / {r['intent']}]")
        print(f"Q: {r['query']}")
        print(f"A: {r['response'][:300]}{'...' if len(r['response']) > 300 else ''}")

    return ds


if __name__ == "__main__":
    try:
        run(llm=llm, tokenizer=tokenizer,                                   # noqa: F821
            snapshot_dataset=snapshot_dataset, query_dataset=query_dataset, # noqa: F821
            generate_snapshot_string=generate_snapshot_string,              # noqa: F821
            hf_token=hf_token)                                              # noqa: F821
    except NameError as e:
        print(f"Missing notebook variable: {e}")
        print("Load llm, tokenizer, snapshot_dataset, query_dataset, "
              "generate_snapshot_string, and hf_token first, then call run(...).")

Pre-formatting 1000 snapshot strings ...
  ✓ 1000 strings ready (avg 6,016 chars each)
Paired 9999 queries to snapshots (each snapshot reused ~10.0x on average).
Resuming: 320 queries already in ./responses_partial.jsonl, skipping those.
To generate: 9,679 queries.
Sample prompt: 3,028 input tokens.


Processed prompts: 100%|██████████| 16/16 [00:20<00:00,  1.25s/it, est. speed input: 2410.65 toks/s, output: 46.23 toks/s]



Sending Batches 0 - 10 to HF


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  train-00030.parquet         : 100%|##########|  139kB /  139kB            

  ✓ https://huggingface.co/datasets/JamesResearch1216/threat-detection-responses-10k
  batch 40/605 done (960/9,999 queries)


Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# **Pass Instructions into the Model**

In [ ]:
from datasets import load_from_disk, Dataset
import json
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# ────────────────────────────────────────────────────────────────────────
# LOAD THE INSTRUCTION DATASET
# ────────────────────────────────────────────────────────────────────────
ds = load_from_disk("./query_instructions_ds")
print(f"Loaded {len(ds)} instruction prompts")
print("Columns:", ds.column_names)

# ────────────────────────────────────────────────────────────────────────
# SETUP MODEL (reuse from your existing notebook)
# ────────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct-AWQ"
llm = LLM(
    model=MODEL_ID,
    quantization="awq_marlin",
    dtype="half",
    gpu_memory_utilization=0.90,
    max_model_len=8192,
    tensor_parallel_size=1,
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()


# ────────────────────────────────────────────────────────────────────────
# STAGE 2: GENERATE QUERY TEMPLATES
# ────────────────────────────────────────────────────────────────────────
sampling_params = SamplingParams(
    temperature=0.8,   # vary it; 0.8-1.0 is good for diversity
    top_p=0.95,
    max_tokens=1024,
)

def make_chat_prompt(instruction, system_prompt):
    """Format instruction + system for Qwen."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# Batch the instructions
instructions = [
    {
        "id": row["id"],
        "instruction": row["instruction"],
        "system_prompt": row["system_prompt"],
        "persona": row["persona"],
        "task": row["task"],
        "modality_scope": row["modality_scope"],
        "placeholders": row["placeholders"],
    }
    for row in ds
]

# Format all as chat prompts
formatted_prompts = [
    make_chat_prompt(inst["instruction"], inst["system_prompt"])
    for inst in instructions
]

# Run inference in batches
batch_size = 1
print(f"Generating {len(formatted_prompts)} instruction prompts through the model (batch_size={batch_size})...")
all_outputs = []

for batch_start in range(0, len(formatted_prompts), batch_size):
    batch_end = min(batch_start + batch_size, len(formatted_prompts))
    batch_prompts = formatted_prompts[batch_start:batch_end]

    print(f"  Batch {batch_start // batch_size + 1}/{(len(formatted_prompts) + batch_size - 1) // batch_size} "
          f"(indices {batch_start}-{batch_end-1})...", end=" ", flush=True)

    batch_outputs = llm.generate(batch_prompts, sampling_params)
    all_outputs.extend(batch_outputs)

    print(f"✓")

outputs = all_outputs
print(f"✓ All {len(outputs)} outputs collected")

# ────────────────────────────────────────────────────────────────────────
# PARSE RESPONSES (the model returns JSON arrays of query templates)
# ────────────────────────────────────────────────────────────────────────
templates_by_id = {}
for i, output in enumerate(outputs):
    response_text = output.outputs[0].text.strip()
    inst_meta = instructions[i]

    # Try to parse as JSON
    try:
        templates = json.loads(response_text)
        if not isinstance(templates, list):
            templates = [templates]
    except json.JSONDecodeError:
        # Fallback: wrap in a list if parsing fails
        print(f"⚠️  ID {inst_meta['id']} ({inst_meta['persona']}/{inst_meta['task']}): "
              f"JSON parse failed, treating response as a single template.")
        templates = [response_text]

    templates_by_id[inst_meta['id']] = {
        "templates": templates,
        "persona": inst_meta["persona"],
        "task": inst_meta["task"],
        "modality_scope": inst_meta["modality_scope"],
        "placeholders": inst_meta["placeholders"],
    }

print(f"Got {len(templates_by_id)} instruction → template responses")

# ────────────────────────────────────────────────────────────────────────
# STAGE 3: EXPAND TEMPLATES INTO CONCRETE QUERIES
# ────────────────────────────────────────────────────────────────────────

def expand_templates(template, task, modality_scope, k_lists=3, max_list_len=5):
    """Turn ONE returned query template into concrete queries by filling placeholders."""
    out = []
    mod_variants = MODALITY_PHRASES.get(modality_scope, [None])

    def fill(s, **kw):
        for key, val in kw.items():
            s = s.replace("{" + key + "}", str(val))
        return s

    if "{idx}" in template:                                   # single sensor -> x23
        for i in range(23):
            for mp in mod_variants:
                out.append(fill(template, idx=i, modality_phrase=mp) if mp
                           else fill(template, idx=i))
    elif "{group}" in template:                               # named groups
        for g in GROUPS:
            for mp in mod_variants:
                out.append(fill(template, group=g, modality_phrase=mp) if mp
                           else fill(template, group=g))
    elif "{indices}" in template:                             # random index subsets
        for _ in range(k_lists):
            picks = sorted(random.sample(range(23), random.randint(2, max_list_len)))
            idx_str = ", ".join(map(str, picks))
            for mp in mod_variants:
                out.append(fill(template, indices=idx_str, modality_phrase=mp) if mp
                           else fill(template, indices=idx_str))
    elif "{modality_phrase}" in template:                     # check_all
        for mp in mod_variants:
            out.append(fill(template, modality_phrase=mp))
    else:
        out.append(template)                                  # already concrete
    return out

all_queries = []

for inst_id, data in templates_by_id.items():
    for template in data["templates"]:
        try:
            expanded = expand_templates(
                template,
                data["task"],
                data["modality_scope"],
                k_lists=3,  # how many random index subsets per template
                max_list_len=5
            )
            for query in expanded:
                all_queries.append({
                    "query": query,
                    "instruction_id": inst_id,
                    "persona": data["persona"],
                    "task": data["task"],
                    "modality_scope": data["modality_scope"],
                    "template": template,  # keep the source template for debugging
                })
        except Exception as e:
            print(f"⚠️  Expansion failed for template '{template[:50]}...': {e}")

print(f"\nExpanded to {len(all_queries)} concrete queries")

# ────────────────────────────────────────────────────────────────────────
# SAVE QUERIES AS AN HF DATASET (local + push to Hub)
# ────────────────────────────────────────────────────────────────────────
query_ds = Dataset.from_list(all_queries)

# Save locally
query_ds.save_to_disk("./queries_ds")
query_ds.to_json("./queries.jsonl")
print(f"✓ Saved locally to ./queries_ds and ./queries.jsonl")

# Push to Hugging Face Hub
HF_USERNAME = "JamesResearch1216"
HF_REPO_NAME = "threat-detection-queries"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f"\nPushing to {HF_REPO_ID}...")
query_ds.push_to_hub(
    HF_REPO_ID,
    token=hf_token,  # uses your existing hf_token from the secret
    private=False,   # set to True if you want it private
)
print(f"✓ Pushed to https://huggingface.co/datasets/{HF_REPO_ID}")

# ────────────────────────────────────────────────────────────────────────
# SUMMARY
# ────────────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"GENERATION COMPLETE")
print(f"{'='*70}")
print(f"Total queries generated: {len(all_queries):,}")
print(f"Queries per task (approx):")
task_counts = {}
for q in all_queries:
    t = q["task"]
    task_counts[t] = task_counts.get(t, 0) + 1
for t in sorted(task_counts.keys()):
    print(f"  {t:<20} {task_counts[t]:>6,}")
print(f"\nQueries per persona (approx):")
persona_counts = {}
for q in all_queries:
    p = q["persona"]
    persona_counts[p] = persona_counts.get(p, 0) + 1
for p in sorted(persona_counts.keys()):
    print(f"  {p:<20} {persona_counts[p]:>6,}")
print(f"\n📊 Dataset at:    https://huggingface.co/datasets/{HF_REPO_ID}")
print(f"{'='*70}")

# Print a few samples
print(f"\nSample queries:")
for q in all_queries[:5]:
    print(f"  [{q['persona']:<20} | {q['task']:<18}] {q['query']}")

Loaded 133 instruction prompts
Columns: ['id', 'persona', 'task', 'modality_scope', 'placeholders', 'n_queries', 'system_prompt', 'instruction']


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 06-18 19:25:07 awq_marlin.py:89] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 06-18 19:25:07 llm_engine.py:213] Initializing an LLM engine (v0.6.0) with config: model='Qwen/Qwen2.5-32B-Instruct-AWQ', speculative_config=None, tokenizer='Qwen/Qwen2.5-32B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq_marlin, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_mod

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Creating parquet from Arrow format:   0%|          | 0/23 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  336kB /  336kB            

✓ Pushed to https://huggingface.co/datasets/JamesResearch1216/threat-detection-queries

GENERATION COMPLETE
Total queries generated: 22,867
Queries per task (approx):
  check_all               571
  multi_sensor_group    5,208
  multi_sensor_list     2,016
  overall_threat           56
  ranking                  56
  single_sensor        14,904
  tasking                  56

Queries per persona (approx):
  air_traffic_controller  3,315
  airport_ops_officer   3,366
  federal_air_marshal   3,382
  first_responder       2,901
  formal_commander      3,366
  incident_commander    3,174
  security_analyst      3,363

📊 Dataset at:    https://huggingface.co/datasets/JamesResearch1216/threat-detection-queries

Sample queries:
  [formal_commander     | overall_threat    ] Report immediate status: is there a confirmed threat in our airspace?
  [formal_commander     | overall_threat    ] Is the system currently identifying any drones as hostile?
  [formal_commander     | overall_threat    ] Pro